In [1]:
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS,utils
from astropy.coordinates import SkyCoord
import astropy.units as u


In [2]:
def findfields(rain,decin):
    with open('fieldcoords.txt') as f:
        lines = f.readlines()
        chunks = [ln.split(' ') for ln in lines]
        fields = [chunk[0] for chunk in chunks]
        layers = [chunk[1] for chunk in chunks]
        corners = [[(float(chunk[3]),float(chunk[2])),(float(chunk[5]),float(chunk[4])),(float(chunk[7]),float(chunk[6])),
                    (float(chunk[9]),float(chunk[8]))] for chunk in chunks]
    fieldsin = []
    for i in range(len(fields)):
        flag=True
        for j in range(4):
            if((decin-corners[i][j%4][0])*(corners[i][(j+1)%4][1]-corners[i][j%4][1]) - 
               (corners[i][(j+1)%4][0]-corners[i][j%4][0])*(rain - corners[i][j%4][1])<0):
                flag=False
                break
        if(flag):
            fieldsin.append([fields[i],int(layers[i])])
    return fieldsin

In [3]:
# List of fields and layers for given coordinates (ra,dec)

fieldsin = findfields(267.17,-30.2)
for el in fieldsin:
    print(el)

['EUC_LE1_VIS-67070-1-C_20250323T112956.000000Z_01_01_01.00.fits.gz', 44]
['EUC_LE1_VIS-67070-1-C_20250323T115516.000000Z_01_01_01.00.fits.gz', 51]
['EUC_LE1_VIS-67070-1-C_20250323T120516.000000Z_01_01_01.00.fits.gz', 44]
['EUC_LE1_VIS-67070-1-C_20250323T123016.000000Z_01_01_01.00.fits.gz', 51]
['EUC_LE1_VIS-67070-1-C_20250323T124016.000000Z_01_01_01.00.fits.gz', 44]
['EUC_LE1_VIS-67070-1-C_20250323T130516.000000Z_01_01_01.00.fits.gz', 51]
['EUC_LE1_VIS-67070-1-C_20250323T131516.000000Z_01_01_01.00.fits.gz', 44]
['EUC_LE1_VIS-67070-1-C_20250323T134016.000000Z_01_01_01.00.fits.gz', 51]


In [6]:
# List of fields, layers and pixels (x,y) for given coordinates (ra,dec)

ra = 269.1
dec = -28.85
fieldsin = findfields(ra,dec)

fieldsout = []
for field in fieldsin:
    filename = field[0]
    print(filename)
    layers = fits.open(filename)
    #Center RA,DEC 
    RA = layers[0].header['RA']
    DEC = layers[0].header['DEC']
    PA = layers[0].header['PA']
    #For one subquadrant, we can modify the wcs to roughly match
    layer = layers[field[1]]
    
    www = WCS(layer.header)
    www.wcs.crval = [RA,DEC]
    www.wcs.cd[0,1] = www.wcs.cd[0,0] *np.cos(np.radians(PA))
    www.wcs.cd[1,0] = www.wcs.cd[0,1]
    www.wcs.cd[1,1] = -www.wcs.cd[0,0]*np.sin(np.radians(PA))
    www.wcs.cd[0,0] = -www.wcs.cd[1,1]
    
    pxy = www.world_to_pixel(SkyCoord(ra=RA*u.degree, dec=DEC*u.degree))
    pxy = www.world_to_pixel(SkyCoord(ra=ra*u.degree, dec=dec*u.degree))
    fieldsout.append(field + [float(pxy[0]),float(pxy[1])])


EUC_LE1_VIS-67070-1-C_20250324T061514.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T062334.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T063154.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T064014.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T065014.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T065834.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T070654.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T071514.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T072514.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T073334.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T074154.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T075014.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T080014.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T080834.000000Z_01_01_01.00.fits.gz
EUC_LE1_VIS-67070-1-C_20250324T081654.000000Z_01_01_01.00.fits.gz
EUC_LE1_VI

In [7]:
fieldsout

[['EUC_LE1_VIS-67070-1-C_20250324T061514.000000Z_01_01_01.00.fits.gz',
  118,
  2051.294118451246,
  923.5927870482865],
 ['EUC_LE1_VIS-67070-1-C_20250324T062334.000000Z_01_01_01.00.fits.gz',
  118,
  1438.8458466537945,
  2038.0535917184175],
 ['EUC_LE1_VIS-67070-1-C_20250324T063154.000000Z_01_01_01.00.fits.gz',
  131,
  1438.8478500944698,
  439.1846901449944],
 ['EUC_LE1_VIS-67070-1-C_20250324T064014.000000Z_01_01_01.00.fits.gz',
  131,
  826.4000775943405,
  1553.650708826788],
 ['EUC_LE1_VIS-67070-1-C_20250324T065014.000000Z_01_01_01.00.fits.gz',
  118,
  2051.29518914186,
  923.5926812065436],
 ['EUC_LE1_VIS-67070-1-C_20250324T065834.000000Z_01_01_01.00.fits.gz',
  118,
  1438.8472133348675,
  2038.0532718980226],
 ['EUC_LE1_VIS-67070-1-C_20250324T070654.000000Z_01_01_01.00.fits.gz',
  131,
  1438.8488518980848,
  439.18431755215533],
 ['EUC_LE1_VIS-67070-1-C_20250324T071514.000000Z_01_01_01.00.fits.gz',
  131,
  826.4013443310157,
  1553.6504826084592],
 ['EUC_LE1_VIS-67070-1-C_